In [1]:
import numpy as np 
import pandas as pd 
from collections import defaultdict
 
import seaborn as sns 
import matplotlib.pyplot as plt 

https://ojs.aaai.org/index.php/AIIDE/article/view/5233/5089

In [16]:
# df_v2 = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\data\entire_odds_stats_2025-11-23.csv')
df_v2 = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\data\ufc_new_rolling.csv')
# df_v2 = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\data\new_combined.csv')
# df_v2 = df_v2.sort_values('date').reset_index(drop=True)
# df_v2 = df_v2[df_v2['winner']!=2].reset_index(drop=True)
# df_v2 = df_v2.dropna().reset_index(drop=True)

In [93]:
def elo_rating(df, k, w90=400):
    
    elo_dic = defaultdict(list)
    red_elo = []
    blue_elo = []
    mu_red_col = []
    mu_blue_col = []
    for _, row in df.iterrows(): 
        red_name = row['fighter_red']
        blue_name = row['fighter_blue']

        if red_name not in elo_dic:
            elo_dic[red_name] = [1500]
        
        if blue_name not in elo_dic:
            elo_dic[blue_name] = [1500]

        prev_blue = elo_dic[blue_name][-1] #elo pre fight
        prev_red = elo_dic[red_name][-1]
        red_elo.append(prev_red)
        blue_elo.append(prev_blue)
        
        if row['winner'] == 1 and pd.notna(row['winner']):

            d = prev_red - prev_blue
            mu_red = 1 / (1 + 10**(-d/w90))
            red_new = prev_red + k * (1-mu_red) #red wins

            d = prev_blue - prev_red
            mu_blue = 1 / (1 + 10**(-d/w90))
            blue_new = prev_blue + k * (0-mu_blue) #blue loses
            
            elo_dic[red_name].append(red_new)
            elo_dic[blue_name].append(blue_new)

        if row['winner'] == 0 and pd.notna(row['winner']):

            d = prev_blue - prev_red
            mu_blue = 1 / (1 + 10**(-d/w90))
            blue_new = prev_blue + k * (1-mu_blue) #blue wins

            d = prev_red - prev_blue
            mu_red = 1 / (1 + 10**(-d/w90))
            red_new = prev_red + k * (0-mu_red) #red loses
            
            elo_dic[red_name].append(red_new)
            elo_dic[blue_name].append(blue_new)
        
        mu_red_col.append(mu_red)
        mu_blue_col.append(mu_blue)

    return np.column_stack([red_elo, blue_elo, mu_red_col, mu_blue_col])

elo_red_blue = elo_rating(df_v2, 50, 400)
df_elo = pd.DataFrame({'elo_red':elo_red_blue[:,0], 'elo_blue':elo_red_blue[:,1],
                       'mu_red':elo_red_blue[:,2], 'mu_blue':elo_red_blue[:,3], 'winner':df_v2['winner']})

df_elo['pred'] = np.where(df_elo['elo_red']>=df_elo['elo_blue'],1,0)
df_elo['pred_mu'] = np.where(df_elo['mu_red']>=df_elo['mu_blue'],df_elo['mu_red'],df_elo['mu_blue'])
df_elo['opp_mu'] = np.where(df_elo['mu_red']<=df_elo['mu_blue'],df_elo['mu_red'],df_elo['mu_blue'])

total_correct = np.sum(np.where(df_elo['pred'] == df_elo['winner'], 1,0))
accuracy = total_correct/df_elo.shape[0]
calibration = df_elo['pred_mu'].sum() / total_correct

y_true = df_elo['winner'].values  
y_pred = df_elo['pred_mu'].values  

log_loss = -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)) / df_elo.shape[0]
df_elo['log_loss'] = log_loss
print(accuracy, calibration, np.sum(log_loss))

0.5618833520669414 0.983093882669956 0.6677407105992023


In [94]:
df_rank = pd.DataFrame({
    'fighter': pd.concat([df_v2['fighter_red'], df_v2['fighter_blue']], ignore_index=True),
    'elo': pd.concat([df_elo['elo_red'], df_elo['elo_blue']], ignore_index=True)
})

# Take the LAST Elo value available (latest fight) for each fighter
df_rank = df_rank.groupby('fighter')['elo'].last().reset_index()

# Sort ranking
df_rank = df_rank.sort_values('elo', ascending=False).reset_index(drop=True)

print(df_rank.head(20))

                  fighter          elo
0               Jon Jones  1808.700477
1       Georges St-Pierre  1802.643660
2            Kamaru Usman  1802.266895
3   Alexander Volkanovski  1787.416240
4            Max Holloway  1785.082403
5        Charles Oliveira  1764.278834
6          Belal Muhammad  1762.702514
7        Robert Whittaker  1753.726318
8          Beneil Dariush  1753.260507
9             Demian Maia  1747.899118
10           Stipe Miocic  1738.499295
11         Daniel Cormier  1735.995516
12      Aljamain Sterling  1731.584549
13         Dustin Poirier  1729.611342
14        Islam Makhachev  1728.300536
15              Jon Fitch  1727.955403
16        Colby Covington  1723.446615
17       Magomed Ankalaev  1721.657651
18          Gilbert Burns  1721.289053
19           Leon Edwards  1721.270086


In [6]:
def map_method_to_score(method):
    method = str(method).lower()
    if "ko" in method or "tko" in method:
        return 1.35
    elif "sub" in method:
        return 1.25
    elif " U-DEC" in method:
        return 1.15
    else:
        return 1.0  # default for unknown / draw

# ----------------------------
# Helper: expected probability
# ----------------------------
def expected_prob(r_a, r_b, scale=400.0):
    # p(A wins) = 1 / (1 + 10^((Rb - Ra)/scale))
    return 1.0 / (1.0 + 10.0 ** ((r_b - r_a) / scale))

# ----------------------------
# MOV scaling functions
# ----------------------------
def mov_linear(w):
    return max(w - 1.0, 1.0)

def mov_log(w):
    # avoid log(0)
    x=np.log(150.0 * max(w - 1.0, 0.0) + 1.0)
    return x

def mov_sqrt(w):
    return np.sqrt(100.0 * max(w, 0.0))

def mov_exp(w):
    return 3.0 ** max(w, 0.0)

MOV_MAP = {
    "linear": mov_linear,
    "log": mov_log,
    "sqrt": mov_sqrt,
    "exp": mov_exp
}

In [99]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.metrics import log_loss, brier_score_loss
from math import log10


# ----------------------------
# Single Elo run function
# ----------------------------
def run_elo_on_matches(matches_df,
                       base_k=20.0,
                       mov_mode="log",
                       cutoff_rating=None,
                       cutoff_k_scale=0.5,
                       w90=None,
                       regress_to_mean=0.0,
                       regress_every_n_matches=None,
                       verbose=False,
                       predict_on=None,
                       scale_override=None):
    """
    Run Elo through matches (chronological) and optionally predict on a holdout set.

    matches_df must include columns:
      - 'date' (chronological order assumed)
      - 'player_a', 'player_b' (ids)
      - 'score_a', 'score_b' (numeric, for margin)
      - 'outcome_a' (1 if A wins, else 0)  OR we compute from score

    Parameters:
      - base_k: base K factor
      - mov_mode: 'linear'|'log'|'sqrt'|'exp'
      - cutoff_rating: rating threshold above which K is scaled
      - cutoff_k_scale: multiplier when above cutoff
      - w90: rating difference corresponding to 90% win prob -> used to compute scale if provided
      - regress_to_mean: fraction to regress ratings toward global mean (0=no regression)
      - regress_every_n_matches: if not None, apply regression every N processed matches
      - predict_on: optional DataFrame of matches to collect predictions for (same schema)
      - scale_override: numeric scale used in expected_prob; if None and w90 provided, compute from w90; else default 400
    Returns:
      - preds (list of predicted probs for rows in predict_on in same order) if predict_on given
      - final_ratings dict
    """
    # copy to avoid editing
    df = matches_df.copy().reset_index(drop=True)

    # compute scale parameter
    if scale_override is not None:
        scale = float(scale_override)
    elif w90 is not None:
        p = 0.9
        denom = log10(p / (1 - p))  # ~= log10(9) ~= 0.9542
        scale = float(w90) / denom
    else:
        scale = 400.0

    mov_func = MOV_MAP.get(mov_mode)
    if mov_func is None:
        raise ValueError("mov_mode must be one of " + ", ".join(MOV_MAP.keys()))

    # ratings store
    ratings = {}
    default_rating = 1500.0
    # optional K per player (kept simple here as constant base_k, but could be per-player)
    # processed count for regression schedule
    processed = 0

    # helper to get rating
    def get_rating(player):
        return ratings.get(player, default_rating)

    # predictions storage if asked
    preds = []
    pred_ids = []

    pre_ratings_red = []
    pre_ratings_blue = []

    # iterate chronologically
    for idx, row in df.iterrows():
        a = row['fighter_red']
        b = row['fighter_blue']
        sa = row.get('winner', None)
        if sa is None:
            print('winner is none')
            sa = 1 if row['score_a'] > row['score_b'] else 0
        sb = 1 - sa

        ra = get_rating(a)
        rb = get_rating(b)
        pre_ratings_red.append(ra)
        pre_ratings_blue.append(rb)

        # expected prob using scale
        pa = expected_prob(ra, rb, scale=scale)

        # margin of victory
        w = abs(row['score_a'] - row['score_b'])
        # ensure w>0 for some functions
        if w <= 0:
            w = 1.0

        k_mult = mov_func(w)

        # apply cutoff scaling if rating above threshold (apply if either or both > cutoff)
        effective_k = base_k * k_mult
        if cutoff_rating is not None:
            # if both above cutoff, scale down (you can change this rule)
            if ra >= cutoff_rating and rb >= cutoff_rating:
                effective_k *= cutoff_k_scale

        # update ratings
        delta = effective_k * (sa - pa)
        ratings[a] = ra + delta
        ratings[b] = rb - delta

        processed += 1
        # optional regression to mean periodically
        if regress_to_mean and regress_every_n_matches and (processed % regress_every_n_matches == 0):
            # regress all ratings toward mean rating
            if len(ratings) > 0:
                mean_rating = np.mean(list(ratings.values()))
                for k in list(ratings.keys()):
                    ratings[k] = ratings[k] + regress_to_mean * (default_rating - ratings[k])

    # If prediction required on a separate DataFrame (e.g., validation set), compute probs using final ratings
    if predict_on is not None:
        preds = []
        for _, row in predict_on.reset_index(drop=True).iterrows():
            a = row['fighter_red']; b = row['fighter_blue']
            ra = ratings.get(a, default_rating)
            rb = ratings.get(b, default_rating)
            p = expected_prob(ra, rb, scale=scale)
            preds.append(p)
        return np.array(preds), ratings

    return None, ratings

# ------------------------------------------------------------
# Cross-validation grid search (expanding window)
# ------------------------------------------------------------
def scope_grid_search(matches_df,
                      param_grid,
                      n_splits=5,
                      metric="logloss",
                      initial_train_frac=0.2,
                      val_window_frac=0.1,
                      verbose=True):
    """
    Grid search for SCOPE-style Elo parameters using expanding time-based CV.

    matches_df:
      chronological DataFrame with columns: 'date' (or already chronological index), 'player_a','player_b','score_a','score_b'
    param_grid: dict of lists, e.g.
       {
         "base_k": [10, 20],
         "mov_mode": ["log", "sqrt"],
         "cutoff_rating": [None, 1800],
         "cutoff_k_scale": [0.5, 1.0],
         "w90": [None, 400],
         "regress_to_mean": [0.0, 0.1],
         "regress_every_n_matches": [None, 1000]
       }
    n_splits: how many expanding folds
    metric: 'logloss' or 'brier'
    Returns:
      DataFrame with param combo and mean CV metric (lower better). Sorted ascending.
    """

    df = matches_df.copy().reset_index(drop=True)
    N = len(df)
    train_start = 0
    results = []

    # build list of candidate param tuples
    keys = list(param_grid.keys())
    combos = list(product(*[param_grid[k] for k in keys]))
    total = len(combos)
    if verbose:
        print(f"Running {total} parameter combinations")

    # prepare split boundaries
    init_train = int(N * initial_train_frac)
    val_window = int(N * val_window_frac)
    if init_train < 10 or val_window < 1:
        raise ValueError("initial_train_frac or val_window_frac too small for dataset size")

    # create split start indices (expanding)
    starts = []
    step = (N - init_train - val_window) // max(1, (n_splits - 1))
    for i in range(n_splits):
        tr_end = init_train + i * step
        val_start = tr_end
        val_end = min(val_start + val_window, N)
        starts.append((0, tr_end, val_start, val_end))

    for combo_idx, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        cv_scores = []

        for (tr0, tr_end, val_start, val_end) in starts:
            if val_end <= val_start or tr_end <= tr0:
                continue
            train_df = df.iloc[tr0:tr_end].reset_index(drop=True)
            val_df = df.iloc[val_start:val_end].reset_index(drop=True)

            # Run Elo on training matches
            _, ratings = run_elo_on_matches(train_df,
                                            base_k=params.get("base_k", 20.0),
                                            mov_mode=params.get("mov_mode", "log"),
                                            cutoff_rating=params.get("cutoff_rating", None),
                                            cutoff_k_scale=params.get("cutoff_k_scale", 1.0),
                                            w90=params.get("w90", None),
                                            regress_to_mean=params.get("regress_to_mean", 0.0),
                                            regress_every_n_matches=params.get("regress_every_n_matches", None),
                                            verbose=False)
            # if params.get("mov_mode", "log") == 'log':
            #     print(params)
            #     print(tr0, tr_end)
            #     print(ratings)

            # Predict probs for validation using learned ratings
            preds = []
            y_true = []
            # compute expected with scale derived from w90 or default 400
            if params.get("w90") is not None:
                p = 0.9
                denom = log10(p / (1 - p))
                scale = float(params.get("w90")) / denom
            else:
                scale = 400.0

            for _, row in val_df.iterrows():
                a = row['fighter_red']; b = row['fighter_blue']
                ra = ratings.get(a, 1500.0)
                rb = ratings.get(b, 1500.0)
                preds.append(expected_prob(ra, rb, scale=scale))
                y_true.append(1 if row.get('winner', None) == 1 else (1 if row['score_a'] > row['score_b'] else 0))

            preds = np.clip(np.array(preds), 1e-12, 1 - 1e-12)
            y_true = np.array(y_true)

            if metric == "logloss":
                sc = log_loss(y_true, preds)
            elif metric == "brier":
                sc = np.mean((preds - y_true) ** 2)
            else:
                raise ValueError("metric must be 'logloss' or 'brier'")

            cv_scores.append(sc)

        if len(cv_scores) == 0:
            mean_score = np.nan
        else:
            mean_score = float(np.mean(cv_scores))

        row = params.copy()
        row['cv_score'] = mean_score
        results.append(row)

        if verbose and combo_idx % max(1, total // 10) == 0:
            print(f"Combo {combo_idx}/{total} done, cv_score={mean_score:.4f}")

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('cv_score').reset_index(drop=True)
    return results_df

# ---------------------------
# Example usage:
# ---------------------------
# matches_df needs columns: date/player_a/player_b/score_a/score_b  (chronological order)

use_method = True  # optional argumen
if use_method and 'method' in df_v2.columns:
    df_v2['score_a'] = df_v2.apply(
    lambda row: map_method_to_score(row['method']) if row['winner'] == 1.0 else 0.0,
    axis=1
    )
    df_v2['score_b'] = df_v2.apply(
        lambda row: map_method_to_score(row['method']) if row['winner'] == 0.0 else 0.0,
        axis=1
    )
else:
    df_v2['score_a'] = df_v2['winner']
    df_v2['score_b'] = 1 - df_v2['winner']

param_grid = {
  "base_k": [50, 35, 75],
  "mov_mode": ["linear","log","sqrt"],
  "cutoff_rating":[None, 1800],
  "cutoff_k_scale":[0.5, 1.0],
  "w90":[None, 400, 325],
  "regress_to_mean":[0.02, 0.05],
  "regress_every_n_matches":[1200, 1000]
}

results_df = scope_grid_search(df_v2, param_grid, n_splits=5, metric="logloss")
print(results_df.head()) 

Running 432 parameter combinations
Combo 43/432 done, cv_score=0.6915
Combo 86/432 done, cv_score=0.7390
Combo 129/432 done, cv_score=1.1382
Combo 172/432 done, cv_score=0.6898
Combo 215/432 done, cv_score=0.7215
Combo 258/432 done, cv_score=0.9016
Combo 301/432 done, cv_score=0.6992
Combo 344/432 done, cv_score=0.7754
Combo 387/432 done, cv_score=1.4771
Combo 430/432 done, cv_score=1.7176
   base_k mov_mode  cutoff_rating  cutoff_k_scale    w90  regress_to_mean  \
0      35   linear            NaN             1.0  400.0             0.05   
1      35   linear         1800.0             0.5  400.0             0.05   
2      35   linear            NaN             0.5  400.0             0.05   
3      35   linear         1800.0             1.0  400.0             0.05   
4      35   linear         1800.0             1.0    NaN             0.05   

   regress_every_n_matches  cv_score  
0                     1000  0.689679  
1                     1000  0.689679  
2                     1000 

In [100]:
results_df.iloc[0]

base_k                           35
mov_mode                     linear
cutoff_rating                   NaN
cutoff_k_scale                  1.0
w90                           400.0
regress_to_mean                0.05
regress_every_n_matches        1000
cv_score                   0.689679
Name: 0, dtype: object

In [ ]:
# _, rating_dict = run_elo_on_matches(df_v2.iloc[0:1006], base_k=75, mov_mode='log', cutoff_rating=None,
#                                                      cutoff_k_scale=.5, w90=None, regress_to_mean=.02,
#                                                      regress_every_n_matches=1200)

_, rating_dict = run_elo_on_matches(df_v2, base_k=130, mov_mode='linear', cutoff_rating=1800.0,
                                                     cutoff_k_scale=.5, w90=250, regress_to_mean=.05,
                                                     regress_every_n_matches=1000)


In [101]:
default_rating = 1500

# def expected_prob(ra, rb):
#     return 1 / (1 + 10 ** ((rb - ra) / 400))

def mov_log(margin):
    """Margin of Victory multiplier (log version)."""
    return np.log2(margin + 2)

def run_elo_with_history(df, base_k=20.0,
                       mov_mode="log",
                       cutoff_rating=None,
                       cutoff_k_scale=0.5,
                       w90=None,
                       regress_to_mean=0.0,
                       regress_every_n_matches=None,
                       verbose=False,
                       predict_on=None,
                       scale_override=None):

    df = df.copy().reset_index(drop=True)
    use_method = True  # optional argumen
    if use_method and 'method' in df.columns:
        df['score_a'] = df.apply(
        lambda row: map_method_to_score(row['method']) if row['winner'] == 1.0 else 0.0,
        axis=1
        )
        df['score_b'] = df.apply(
            lambda row: map_method_to_score(row['method']) if row['winner'] == 0.0 else 0.0,
            axis=1
        )
    else:
        print("ELO SCOPE ERROR")
        df['score_a'] = df['winner']
        df['score_b'] = 1 - df['winner']

    # compute scale parameter
    if scale_override is not None:
        scale = float(scale_override)
    elif w90 is not None:
        p = 0.9
        denom = log10(p / (1 - p))  # ~= log10(9) ~= 0.9542
        scale = float(w90) / denom
    else:
        scale = 400.0

    mov_func = MOV_MAP.get(mov_mode)
    if mov_func is None:
        raise ValueError("mov_mode must be one of " + ", ".join(MOV_MAP.keys()))

    # ratings store
    ratings = {}
    default_rating = 1500.0
    # optional K per player (kept simple here as constant base_k, but could be per-player)
    # processed count for regression schedule
    processed = 0

    # helper to get rating
    def get_rating(player):
        return ratings.get(player, default_rating)

    # predictions storage if asked
    preds = []
    pred_ids = []

    pre_ratings_red = []
    pre_ratings_blue = []
    fighter_red = []
    fighter_blue = []
    dates = []
    # iterate chronologically
    for idx, row in df.iterrows():
        dates.append(row['date'])
        a = row['fighter_red']
        b = row['fighter_blue']
        sa = row.get('winner', None)
        if sa is None:
            print('winner is none')
            sa = 1 if row['score_a'] > row['score_b'] else 0
        sb = 1 - sa

        ra = get_rating(a)
        rb = get_rating(b)

        fighter_red.append(a)
        fighter_blue.append(b)
        pre_ratings_red.append(ra)
        pre_ratings_blue.append(rb)
        
        # expected prob using scale
        pa = expected_prob(ra, rb, scale=scale)

        # margin of victory
        w = abs(row['score_a'] - row['score_b'])
        # ensure w>0 for some functions
        if w <= 0:
            w = 1.0

        k_mult = mov_func(w)

        # apply cutoff scaling if rating above threshold (apply if either or both > cutoff)
        effective_k = base_k * k_mult
        if cutoff_rating is not None:
            # if both above cutoff, scale down (you can change this rule)
            if ra >= cutoff_rating and rb >= cutoff_rating:
                effective_k *= cutoff_k_scale

        # update ratings
        delta = effective_k * (sa - pa)
        ratings[a] = ra + delta
        ratings[b] = rb - delta

        processed += 1
        # optional regression to mean periodically
        if regress_to_mean and regress_every_n_matches and (processed % regress_every_n_matches == 0):
            # regress all ratings toward mean rating
            if len(ratings) > 0:
                mean_rating = np.mean(list(ratings.values()))
                for k in list(ratings.keys()):
                    ratings[k] = ratings[k] + regress_to_mean * (default_rating - ratings[k])


    # If prediction required on a separate DataFrame (e.g., validation set), compute probs using final ratings
    if predict_on is not None:
        preds = []
        for _, row in predict_on.reset_index(drop=True).iterrows():
            a = row['fighter_red']; b = row['fighter_blue']
            ra = ratings.get(a, default_rating)
            rb = ratings.get(b, default_rating)
            p = expected_prob(ra, rb, scale=scale)
            preds.append(p)
        return np.array(preds), ratings


    ratings_over_time = {}
    # # Save history
    # for f in [red, blue]:
    #     if f not in ratings_over_time:
    #         ratings_over_time[f] = []
    #     ratings_over_time[f].append((idx, ratings[f]))

    return ratings, ratings_over_time, pd.DataFrame({'elo_red':pre_ratings_red, 'elo_blue':pre_ratings_blue, 'fighter_red':fighter_red, 'fighter_blue':fighter_blue, 'date':dates})

params = {'base_k':35, 'mov_mode':'linear', 'cutoff_rating':None, 'cutoff_k_scale':1, 'w90':400, 'regress_to_mean':.05, 'regress_every_n_matches':1000}

df_v2 = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\data\ufc_new_rolling.csv')
ratings, ratings_over_time, df_rating_fighter = run_elo_with_history(df_v2, **params)

ratings_df = pd.DataFrame(list(ratings.items()), columns=['fighter', 'rating'])
ratings_df = ratings_df.sort_values('rating', ascending=False)
top_fighters = ratings_df.head(40)['fighter'].tolist()
# plt.figure(figsize=(12,6))
# for f in top_fighters:
#     idxs, vals = zip(*ratings_over_time[f])
#     plt.plot(idxs, vals, label=f)

# plt.xlabel('Match Index')
# plt.ylabel('Elo Rating')
# plt.title('Elo Ratings Over Time (Top 5 Fighters)')
# plt.legend()
# plt.show()

In [102]:
ratings_df.head(20)

,fighter,rating
1184,Islam Makhachev,1710.105041
334,Jon Jones,1692.205520
1492,Merab Dvalishvili,1681.915204
1103,Leon Edwards,1669.371768
1638,Grant Dawson,1668.496535
13,Georges St-Pierre,1665.177411
675,Max Holloway,1663.985948
1466,Muslim Salikhov,1662.784038
1807,Tom Aspinall,1662.091158
1201,Kamaru Usman,1660.502296


In [103]:
df_rating_fighter['winner'] = df_v2['winner']
df_rating_fighter['choice_elo'] = np.where(df_rating_fighter['elo_red'] > df_rating_fighter['elo_blue'], 1, 0)

df_rating_fighter['choice_correct'] = np.where(df_rating_fighter['choice_elo']==df_rating_fighter['winner'], 1, 0)
print(df_rating_fighter['choice_correct'].mean())


0.5316597976770326


In [65]:
max(ratings.values())

1864.1274608588526

In [96]:
fighters_red = df_rating_fighter[['fighter_red', 'elo_red','date']].rename(columns={'fighter_red': 'fighter', 'elo_red': 'elo'})
fighters_blue = df_rating_fighter[['fighter_blue', 'elo_blue', 'date']].rename(columns={'fighter_blue': 'fighter', 'elo_blue': 'elo'})
print(fighters_blue.columns)

all_fighters = pd.concat([fighters_red, fighters_blue], ignore_index=True)
all_fighters = all_fighters.sort_values('date')   # or whatever your chronological column is

top_fighters = all_fighters.groupby('fighter')['elo'].last().sort_values(ascending=False)


top_10_fighters = top_fighters.head(50)
top_10_fighters


Index(['fighter', 'elo', 'date'], dtype='object')


fighter
Islam Makhachev          1816.650902
Leon Edwards             1801.555494
Demetrious Johnson       1791.271609
Daniel Cormier           1788.959072
Georges St-Pierre        1787.230977
Merab Dvalishvili        1771.797503
Jon Jones                1767.665175
Khabib Nurmagomedov      1759.627898
Dricus Du Plessis        1749.787682
Magomed Ankalaev         1745.040360
Nassourdine Imavov       1740.245827
Belal Muhammad           1739.975579
Grant Dawson             1735.412124
Max Holloway             1726.492000
Jan Blachowicz           1725.355508
Ilia Topuria             1724.177744
Amanda Nunes             1723.945982
Kamaru Usman             1723.838333
Khamzat Chimaev          1723.371918
Francis Ngannou          1721.100946
Stipe Miocic             1718.825595
Aljamain Sterling        1711.927266
Alexander Volkanovski    1711.662255
Charles Oliveira         1709.607414
Alex Pereira             1708.557946
Rinat Fakhretdinov       1707.566787
Alexandre Pantoja        1707.

In [98]:
ratings_df.head()

,fighter,rating
1184,Islam Makhachev,1816.295846
1492,Merab Dvalishvili,1778.034851
334,Jon Jones,1776.758482
1807,Tom Aspinall,1758.749993
1466,Muslim Salikhov,1751.747045


In [10]:
search_str = "anderson"

matching_rows = ratings_df[ratings_df['fighter'].str.contains(search_str, case=False, na=False)]
matching_rows

,fighter,rating
0,anderson silva,1550.593670
666,corey anderson,1548.184943
1385,joanderson brito,1517.996276
1025,megan anderson,1504.853834
1074,anderson dos santos,1482.769724
